# Transform and Normalize Governance Data

This notebook combines Purview and Unified Catalog extracts into curated tables for reporting and Fabric loading.

Outputs:
- data/processed/dim_collections
- data/processed/dim_sources
- data/processed/dim_assets
- data/processed/dim_data_products
- data/processed/fact_scans
- data/processed/fact_quality
- data/processed/relationships

In [ ]:
import json
import logging
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, IntegerType, StringType, StructField, StructType

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

spark = SparkSession.builder.appName("PurviewTransformData").getOrCreate()
root = Path.cwd().parent
raw_root = root / "data" / "raw"
processed_root = root / "data" / "processed"
reports_root = root / "data" / "reports"

for directory in (raw_root, processed_root, reports_root):
    directory.mkdir(parents=True, exist_ok=True)

def load_payload(file_name: str, global_name: str) -> Dict[str, Any]:
    global_value = globals().get(global_name)
    if isinstance(global_value, dict):
        return global_value

    file_path = raw_root / file_name
    if not file_path.exists():
        logger.warning(f"Payload file not found: {file_path}")
        return {}

    with file_path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def records(payload: Dict[str, Any], key: str) -> List[Dict[str, Any]]:
    value = (payload or {}).get(key, [])
    return value if isinstance(value, list) else []

def create_df(record_list: List[Dict[str, Any]], schema: StructType):
    if record_list:
        return spark.createDataFrame(record_list)
    return spark.createDataFrame([], schema)

purview_payload = load_payload("purview_extraction.json", "purview_extraction_data")
catalog_payload = load_payload("unified_catalog_extraction.json", "unified_catalog_data")
setup_payload = load_payload("setup_audit_config.json", "audit_config")

collections = records(purview_payload, "collections")
sources = records(purview_payload, "sources")
purview_assets = records(purview_payload, "assets")
scans = records(purview_payload, "scans")
catalog_products = records(catalog_payload, "data_products")
catalog_assets = records(catalog_payload, "catalog_assets")
quality_scores = records(catalog_payload, "data_quality")
domains = records(catalog_payload, "domains")

print(json.dumps({
    "collections": len(collections),
    "sources": len(sources),
    "purview_assets": len(purview_assets),
    "scans": len(scans),
    "catalog_products": len(catalog_products),
    "catalog_assets": len(catalog_assets),
    "quality_scores": len(quality_scores),
    "domains": len(domains)
}, indent=2))

In [ ]:
collection_schema = StructType([
    StructField("collection_id", StringType(), True),
    StructField("collection_name", StringType(), True),
    StructField("parent_collection_id", StringType(), True),
    StructField("friendly_name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("source_account", StringType(), True),
    StructField("is_primary", BooleanType(), True),
    StructField("extracted_at", StringType(), True),
])
source_schema = StructType([
    StructField("source_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("source_type", StringType(), True),
    StructField("endpoint", StringType(), True),
    StructField("collection_id", StringType(), True),
    StructField("scan_count", IntegerType(), True),
    StructField("source_account", StringType(), True),
    StructField("is_primary", BooleanType(), True),
])
asset_schema = StructType([
    StructField("asset_id", StringType(), True),
    StructField("asset_name", StringType(), True),
    StructField("asset_type", StringType(), True),
    StructField("collection_id", StringType(), True),
    StructField("data_product_id", StringType(), True),
    StructField("data_product_name", StringType(), True),
    StructField("owner", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("source_account", StringType(), True),
    StructField("is_primary", BooleanType(), True),
    StructField("certified", BooleanType(), True),
])
data_product_schema = StructType([
    StructField("data_product_id", StringType(), True),
    StructField("data_product_name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("owner", StringType(), True),
    StructField("domain_id", StringType(), True),
    StructField("created_date", StringType(), True),
    StructField("asset_count", IntegerType(), True),
])
scan_schema = StructType([
    StructField("scan_id", StringType(), True),
    StructField("scan_name", StringType(), True),
    StructField("data_source_id", StringType(), True),
    StructField("scan_status", StringType(), True),
    StructField("last_run", StringType(), True),
    StructField("source_account", StringType(), True),
])
quality_schema = StructType([
    StructField("asset_id", StringType(), True),
    StructField("asset_name", StringType(), True),
    StructField("quality_score", StringType(), True),
    StructField("quality_status", StringType(), True),
    StructField("data_product_id", StringType(), True),
    StructField("last_evaluated", StringType(), True),
])
relationship_schema = StructType([
    StructField("relationship_type", StringType(), True),
    StructField("parent_id", StringType(), True),
    StructField("child_id", StringType(), True),
    StructField("parent_name", StringType(), True),
    StructField("child_name", StringType(), True),
    StructField("source_system", StringType(), True),
])

dim_collections_df = create_df(collections, collection_schema).dropDuplicates(["collection_id", "source_account"])
dim_sources_df = create_df(sources, source_schema).dropDuplicates(["source_id", "source_account"])
dim_data_products_df = create_df(catalog_products, data_product_schema).dropDuplicates(["data_product_id"])
fact_scans_df = create_df(scans, scan_schema).dropDuplicates(["scan_id", "source_account"])
fact_quality_df = create_df(quality_scores, quality_schema).dropDuplicates(["asset_id", "data_product_id"])

normalized_assets = []
for asset in purview_assets:
    normalized_assets.append({
        "asset_id": asset.get("asset_id"),
        "asset_name": asset.get("asset_name"),
        "asset_type": asset.get("asset_type"),
        "collection_id": asset.get("collection_id"),
        "data_product_id": None,
        "data_product_name": None,
        "owner": asset.get("owner"),
        "source_system": "purview",
        "source_account": asset.get("source_account"),
        "is_primary": asset.get("is_primary"),
        "certified": False,
    })

for asset in catalog_assets:
    normalized_assets.append({
        "asset_id": asset.get("asset_id"),
        "asset_name": asset.get("asset_name"),
        "asset_type": asset.get("asset_type"),
        "collection_id": None,
        "data_product_id": asset.get("data_product_id"),
        "data_product_name": asset.get("data_product_name"),
        "owner": asset.get("owner"),
        "source_system": "fabric",
        "source_account": None,
        "is_primary": None,
        "certified": asset.get("certified", False),
    })

dim_assets_df = create_df(normalized_assets, asset_schema).dropDuplicates(["asset_id", "source_system"])

relationship_records = []
for collection in collections:
    if collection.get("parent_collection_id"):
        relationship_records.append({
            "relationship_type": "collection_parent",
            "parent_id": collection.get("parent_collection_id"),
            "child_id": collection.get("collection_id"),
            "parent_name": None,
            "child_name": collection.get("collection_name"),
            "source_system": "purview",
        })

for source in sources:
    if source.get("collection_id"):
        relationship_records.append({
            "relationship_type": "collection_source",
            "parent_id": source.get("collection_id"),
            "child_id": source.get("source_id"),
            "parent_name": None,
            "child_name": source.get("source_name"),
            "source_system": "purview",
        })

for asset in purview_assets:
    if asset.get("collection_id"):
        relationship_records.append({
            "relationship_type": "collection_asset",
            "parent_id": asset.get("collection_id"),
            "child_id": asset.get("asset_id"),
            "parent_name": None,
            "child_name": asset.get("asset_name"),
            "source_system": "purview",
        })

for scan in scans:
    if scan.get("data_source_id"):
        relationship_records.append({
            "relationship_type": "source_scan",
            "parent_id": scan.get("data_source_id"),
            "child_id": scan.get("scan_id"),
            "parent_name": None,
            "child_name": scan.get("scan_name"),
            "source_system": "purview",
        })

for asset in catalog_assets:
    if asset.get("data_product_id"):
        relationship_records.append({
            "relationship_type": "data_product_asset",
            "parent_id": asset.get("data_product_id"),
            "child_id": asset.get("asset_id"),
            "parent_name": asset.get("data_product_name"),
            "child_name": asset.get("asset_name"),
            "source_system": "fabric",
        })

for product in catalog_products:
    if product.get("domain_id"):
        relationship_records.append({
            "relationship_type": "domain_data_product",
            "parent_id": product.get("domain_id"),
            "child_id": product.get("data_product_id"),
            "parent_name": None,
            "child_name": product.get("data_product_name"),
            "source_system": "fabric",
        })

relationships_df = create_df(relationship_records, relationship_schema).dropDuplicates()

display(dim_assets_df)
display(relationships_df)

In [ ]:
datasets = {
    "dim_collections": dim_collections_df,
    "dim_sources": dim_sources_df,
    "dim_assets": dim_assets_df,
    "dim_data_products": dim_data_products_df,
    "fact_scans": fact_scans_df,
    "fact_quality": fact_quality_df,
    "relationships": relationships_df,
}

for dataset_name, dataset_df in datasets.items():
    target_path = processed_root / dataset_name
    dataset_df.write.mode("overwrite").parquet(str(target_path))
    print(f"Saved {dataset_name} to {target_path}")

transform_summary = {
    "generated_at": datetime.utcnow().isoformat(),
    "workspace": setup_payload.get("fabric_workspace", {}),
    "counts": {name: df.count() for name, df in datasets.items()},
}

summary_path = reports_root / "transform_summary.json"
with summary_path.open("w", encoding="utf-8") as handle:
    json.dump(transform_summary, handle, indent=2)

print(json.dumps(transform_summary, indent=2))